# Notebook de Aplicação e Logging de Previsões

**Objetivo:** Carregar os modelos CatBoost, aplicar aos dados mais recentes disponíveis e salvar as previsões em um log histórico (CSV) para análise e acompanhamento.

**Fluxo de Execução:**
1.  **Executar `pipeline_gerar_dados_recentes.ipynb`:** Garante que o arquivo `aluminium_full_featured_retrained.csv` está atualizado com as últimas cotações e features.
2.  **Executar este notebook:** Ele pegará a última linha do arquivo de dados, gerará as previsões para todos os horizontes e adicionará os resultados ao arquivo `log_previsoes.csv`.

Rodar pipeline_gerar_dados_recentes.ipynb: Para baixar os dados mais recentes e atualizar o arquivo aluminium_full_featured_retrained.csv.

Rodar aplicar_modelo.ipynb: Ele vai pegar a última linha de dados, gerar as 4 novas previsões e, como já confirmamos, adicionar (append) essas novas linhas ao seu log_previsoes.csv, continuando o histórico que você começou com a simulação.

Rodar validar_previsoes.ipynb (quando quiser): Este notebook já está pronto para o novo fluxo. Ele vai ler o log_previsoes.csv completo (com dados simulados e ao vivo), verificar todas as previsões que já "venceram" (cujo horizonte já passou) e gerar o relatório de performance atualizado.

In [23]:
# --- Célula de Instalação e Importação de Bibliotecas ---
!pip install -q pandas numpy catboost joblib

import pandas as pd
import joblib
import os
import json
from datetime import datetime

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [24]:
# --- Célula de Configuração de Variáveis e Carregamento de Dados ---

DATA_FILE = 'aluminium_full_featured_retrained.csv'
MODELS_DIR = 'models/' # Adapte se os modelos estiverem em outra pasta
LOG_FILE = 'log_previsoes.csv' # Arquivo que armazenará o histórico
horizons = [1, 5, 30, 90]

# Carrega o dataset com features
try:
    df_features = pd.read_csv(DATA_FILE, parse_dates=['date'])
    # Pega a última linha de dados, que é a mais recente para previsão
    latest_data = df_features.sort_values('date', ascending=False).iloc[0:1]
    data_base_previsao = latest_data['date'].dt.date.iloc[0]
    print(f"Dados de {data_base_previsao} carregados para gerar as previsões.")
except FileNotFoundError:
    print(f"ERRO: Arquivo '{DATA_FILE}' não encontrado. Execute o pipeline de dados primeiro.")
    latest_data = None

Dados de 2025-09-15 carregados para gerar as previsões.


In [25]:
# --- Célula de Geração e Logging das Previsões (CORRIGIDO) ---

if latest_data is not None:
    # Lista para armazenar os resultados da execução atual
    current_run_logs = []
    data_execucao = datetime.now()

    # Prepara a base das features que é comum a todos modelos
    base_feature_cols = [col for col in df_features.columns if col not in ['date'] and not any(s in col for s in ['bin_', 'ret_', 'future_'])]
    X_predict_base = latest_data[base_feature_cols].copy()

    print("--- Iniciando Previsões ---")
    for h in horizons:
        try:
            # Carrega os artefatos específicos do horizonte
            model = joblib.load(os.path.join(MODELS_DIR, f'catboost_model_horizon_{h}d_retrained.joblib'))
            encoder = joblib.load(os.path.join(MODELS_DIR, f'label_encoder_horizon_{h}d_retrained.joblib'))
            
            # Adiciona a feature 'horizon' que o modelo espera
            X_predict = X_predict_base.copy()
            X_predict['horizon'] = h
            
            # Garante a ordem correta das colunas
            X_predict = X_predict[model.feature_names_]

            # Gera predição e probabilidades
            pred_encoded = model.predict(X_predict)[0]
            pred_proba = model.predict_proba(X_predict)[0]
            pred_class = encoder.inverse_transform([pred_encoded])[0]
            proba_dict = {encoder.inverse_transform([i])[0]: p for i, p in enumerate(pred_proba)}
            
            # Adiciona o resultado à lista de logs com o nome de coluna padronizado
            current_run_logs.append({
                'data_execucao_simulada': data_execucao.strftime('%Y-%m-%d %H:%M:%S'), # <--- CORREÇÃO APLICADA AQUI
                'data_base_previsao': data_base_previsao,
                'horizonte_dias': h,
                'previsao_classe': pred_class,
                'probabilidades': json.dumps(proba_dict) # Salva como string JSON
            })
            print(f"  - Horizonte {h} dias: {pred_class} (Probabilidades: {proba_dict})")

        except Exception as e:
            print(f"ERRO ao processar o horizonte {h}: {e}")

    # --- Salvando os Logs em CSV ---
    if current_run_logs:
        log_df = pd.DataFrame(current_run_logs)
        
        # Se o arquivo de log já existe, anexa os novos dados. Senão, cria um novo.
        if os.path.exists(LOG_FILE):
            log_df.to_csv(LOG_FILE, mode='a', header=False, index=False)
            print(f"\nPrevisões adicionadas ao log existente: '{LOG_FILE}'")
        else:
            log_df.to_csv(LOG_FILE, mode='w', header=True, index=False)
            print(f"\nNovo arquivo de log de previsões criado: '{LOG_FILE}'")
        
        # Exibe o log completo
        print("\n--- Histórico de Previsões ---")
        display(pd.read_csv(LOG_FILE))

--- Iniciando Previsões ---
  - Horizonte 1 dias: Na mesma (Probabilidades: {'Cai': 0.1635093981858415, 'Cai muito': 0.10531929796479364, 'Na mesma': 0.5158347375424782, 'Sobe': 0.18368661068294823, 'Sobe muito': 0.0316499556239384})
  - Horizonte 5 dias: Cai muito (Probabilidades: {'Cai': 0.18200829176653516, 'Cai muito': 0.45464665382061037, 'Na mesma': 0.28757811936867944, 'Sobe': 0.06901253706184184, 'Sobe muito': 0.006754397982333217})
  - Horizonte 30 dias: Cai muito (Probabilidades: {'Cai': 0.012785290283582301, 'Cai muito': 0.8438183916378839, 'Na mesma': 0.03776399522265755, 'Sobe': 0.08337265414628313, 'Sobe muito': 0.022259668709593266})
  - Horizonte 90 dias: Cai muito (Probabilidades: {'Cai': 0.048674840433108685, 'Cai muito': 0.8869175739250942, 'Na mesma': 0.020393332687061096, 'Sobe': 0.01982237914531297, 'Sobe muito': 0.02419187380942333})

Previsões adicionadas ao log existente: 'log_previsoes.csv'

--- Histórico de Previsões ---


,data_execucao_simulada,data_base_previsao,horizonte_dias,previsao_classe,probabilidades
0,2025-09-01,2025-08-29,1,Cai muito,"{""Cai"": 0.005388586587980231, ""Cai muito"": 0.9..."
1,2025-09-01,2025-08-29,5,Cai muito,"{""Cai"": 0.01666639234702933, ""Cai muito"": 0.95..."
2,2025-09-01,2025-08-29,30,Cai muito,"{""Cai"": 0.013962355534113857, ""Cai muito"": 0.9..."
3,2025-09-01,2025-08-29,90,Cai muito,"{""Cai"": 0.006521856777931138, ""Cai muito"": 0.9..."
4,2025-09-02,2025-09-01,1,Na mesma,"{""Cai"": 0.1625895073099922, ""Cai muito"": 0.282..."
...,...,...,...,...,...
63,2025-09-16,2025-09-15,90,Cai muito,"{""Cai"": 0.048674840433108685, ""Cai muito"": 0.8..."
64,2025-09-16 10:54:25,2025-09-15,1,Na mesma,"{""Cai"": 0.1635093981858415, ""Cai muito"": 0.105..."
65,2025-09-16 10:54:25,2025-09-15,5,Cai muito,"{""Cai"": 0.18200829176653516, ""Cai muito"": 0.45..."
66,2025-09-16 10:54:25,2025-09-15,30,Cai muito,"{""Cai"": 0.012785290283582301, ""Cai muito"": 0.8..."
